# 07 · Type hints, docstrings and dataclasses

**Day 2 · about 75 minutes live**

Yesterday you wrote classes that work. Today is about code that *other people* can use correctly without
asking you, and that fails early when someone uses it wrongly. "Other people" includes you in three months.

This notebook covers three tools that work together:

- **Type hints** say what kind of value a function expects and returns. Your editor and `mypy` use them to
  catch mistakes before the code runs.
- **Docstrings** say what a function or class is for, what its arguments mean and what can go wrong.
- **Dataclasses** turn a bag of settings (a config dict full of magic numbers) into a small class with
  defaults, a readable `repr`, and validation in one place.

**Learning goals.** By the end you can ...

- read type hints in library code and explain what they promise and what they do not `[Understand, level 2]`
- add type hints and Google-style docstrings to your own functions and classes `[Apply, level 2]`
- replace a configuration dict with a validated, immutable dataclass `[Apply, level 2]`
- decide between a dataclass and Pydantic for a given piece of configuration `[Analyse, level 3]`

**Before you start:** notebooks 01 (classes, `@classmethod`) and 04 (properties). Notebook 03 helps for
`__repr__` and `__eq__`.

Exercises come in three levels: **Core** (everyone), **Stretch** (if you finish early), **Challenge** (for
experienced programmers). Check cells fail until you solve the exercise above them; that is expected.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

DATA_PATH = Path("../data/raw/telco_churn.csv")
CONFIG_PATH = Path("../configs/baseline.json")

customers = pd.read_csv(DATA_PATH)
customers["Churn"] = (customers["Churn"] == "Yes").astype(int)
NUMERIC = ["tenure", "MonthlyCharges", "SeniorCitizen"]
X, y = customers[NUMERIC], customers["Churn"]
print(X.shape, f"churn rate {y.mean():.1%}")

## 1. The problem: magic numbers and a config dict with a typo

In the warm-up notebook every setting was a number typed directly into a cell: `test_size=0.2`,
`random_state=42`, `n_estimators=100`. The usual next step is to collect them in a dict, which is
better, but has a nasty failure mode. Look closely at the key below.

In [ ]:
config = {
    "model": "random_forest",
    "n_estimatros": 500,  # typo
    "test_size": 0.2,
    "random_state": 42,
}


def train(config: dict) -> RandomForestClassifier:
    X_train, _, y_train, _ = train_test_split(
        X, y, test_size=config.get("test_size", 0.25), random_state=config.get("random_state", 0)
    )
    model = RandomForestClassifier(
        n_estimators=config.get("n_estimators", 100), random_state=config.get("random_state", 0)
    )
    return model.fit(X_train, y_train)


model = train(config)
print("Trees actually trained:", model.n_estimators)

We asked for 500 trees and silently got 100. Nothing crashed, no warning appeared, and the experiment log
says "500". A dict accepts any key, and `.get()` with a default hides every typo.

What we want instead:

1. a fixed list of allowed settings, so a typo is an error;
2. defaults written down once, in one place;
3. invalid values (a `test_size` of 1.5) rejected when the config is created, not halfway through training;
4. a description of what each setting means, right next to it.

Type hints, docstrings and dataclasses give us all four. We start with type hints because dataclasses are
built on them.

## 2. Type hints: the essentials

A **type hint** (or annotation) is written after a parameter name with a colon, and after the parameter
list with an arrow for the return value.

In [ ]:
def recall(tp: int, fn: int) -> float:
    return tp / (tp + fn)


def churn_rate_by(df: pd.DataFrame, column: str) -> pd.Series:
    return df.groupby(column)["Churn"].mean().sort_values(ascending=False)


print(recall(tp=300, fn=74))
churn_rate_by(customers, "Contract")

The patterns you will meet every day:

| Hint | Meaning |
|---|---|
| `int`, `float`, `str`, `bool` | built-in types |
| `list[str]` | a list whose items are strings, e.g. feature names |
| `dict[str, float]` | keys are strings, values are floats, e.g. `{"recall": 0.81}` |
| `tuple[str, ...]` | a tuple of strings of any length |
| `pd.DataFrame`, `np.ndarray` | classes from libraries work too |
| `float \| None` | a float or `None` (Python 3.10+). Use it whenever `None` is allowed |
| `Callable[[pd.Series, np.ndarray], float]` | a function that takes those two arguments and returns a float |

You will also see the older spelling from the `typing` module in existing code: `List[str]`,
`Dict[str, float]`, `Optional[float]` (same as `float | None`), `Union[int, str]` (same as `int | str`).
They mean the same thing. Prefer the new spelling in new code.

In [ ]:
from collections.abc import Callable

from sklearn.metrics import f1_score, recall_score


def evaluate(
    model: RandomForestClassifier,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    metric: Callable[[pd.Series, np.ndarray], float] = recall_score,
    threshold: float | None = None,
) -> float:
    if threshold is None:
        y_pred = model.predict(X_test)
    else:
        y_pred = (model.predict_proba(X_test)[:, 1] >= threshold).astype(int)
    return metric(y_test, y_pred)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=42).fit(
    X_train, y_train
)
print(f"recall at default threshold: {evaluate(rf, X_test, y_test):.2f}")
print(f"recall at threshold 0.3:     {evaluate(rf, X_test, y_test, threshold=0.3):.2f}")
print(
    f"f1 at threshold 0.3:         {evaluate(rf, X_test, y_test, metric=f1_score, threshold=0.3):.2f}"
)

### What type hints do *not* do

Python itself ignores type hints when it runs your code. They are documentation that tools can read.

In [ ]:
print(recall(tp=300.5, fn=74))  # a float where we promised an int: Python does not complain
print(recall.__annotations__)  # the hints are just stored on the function

Two kinds of tools read them:

- **Your editor** (VS Code with Pylance) uses them for autocompletion and underlines suspicious calls.
- **mypy**, a static type checker, reads a whole file without running it and reports every mismatch.
  You run it from the terminal. Suppose `metrics.py` contains:

```python
def recall(tp: int, fn: int) -> float:
    return tp / (tp + fn)


def churn_rate(labels: list[int]) -> float:
    return sum(labels) / len(labels)


score = recall(30, "12")
rate = churn_rate([1, 0, 0, None])
message: str = churn_rate([1, 0])
```

```text
$ uv run mypy metrics.py
metrics.py:9: error: Argument 2 to "recall" has incompatible type "str"; expected "int"  [arg-type]
metrics.py:10: error: List item 3 has incompatible type "None"; expected "int"  [list-item]
metrics.py:11: error: Incompatible types in assignment (expression has type "float", variable has type "str")  [assignment]
Found 3 errors in 1 file (checked 1 source file)
```

All three are real bugs, found without running anything. This repo is set up so that `uv run mypy`
checks the reference package in `src/churn`.

If you want checks **at runtime** (for data coming from outside: a config file, an API request), you need
a validation library such as Pydantic. See section 5.

### Check your understanding

What is wrong with this signature, even though the code runs?

```python
def flag_customers(probabilities: np.ndarray, threshold: float = None) -> np.ndarray:
    ...
```

<details><summary>Answer</summary>

The default is `None`, but the hint says `float`. A reader (and mypy) will assume `threshold` is always a
number and skip the `None` case. Write `threshold: float | None = None` and handle `None` explicitly inside
the function. The same goes for `features: list[str] = None`, which you will find in a lot of code.
</details>

## 3. Docstrings in Google style

A **docstring** is a string literal placed as the first statement of a module, class or function. Python
stores it in `__doc__`, `help()` prints it, and your editor shows it when you hover over a name.

Type hints say *what type*. The docstring says *what it means*: units, allowed ranges, what is returned,
which errors can be raised and why. Do not repeat what the hints already say.

There are several conventions (NumPy style is common in scientific libraries, reStructuredText in older
code). We use **Google style**, which is easy to read in the source and supported by documentation tools.

In [ ]:
def flag_customers(probabilities: np.ndarray, threshold: float = 0.5) -> np.ndarray:
    """Turn churn probabilities into a list of customers to contact.

    The retention team contacts every flagged customer, so a lower threshold
    means more offers (higher cost) but fewer missed churners.

    Args:
        probabilities: Predicted churn probability per customer, between 0 and 1.
        threshold: Probability from which a customer is flagged. Must be
            strictly between 0 and 1.

    Returns:
        Array of 0/1 flags with the same length as ``probabilities``.

    Raises:
        ValueError: If ``threshold`` is not strictly between 0 and 1.
    """
    if not 0 < threshold < 1:
        raise ValueError(f"threshold must be between 0 and 1, got {threshold}")
    return (probabilities >= threshold).astype(int)


help(flag_customers)

For a class, the class docstring describes what an instance represents and lists its attributes. Methods
get their own docstrings. Private helpers (`_something`) often only need a one-line docstring or none.

Rules of thumb:

- First line: one sentence saying what it does, in the imperative ("Turn ...", "Return ...").
- Document units and ranges ("months", "between 0 and 1"), they are invisible in the types.
- Document `Raises:` for errors a caller should expect and handle.
- If the name and hints already say everything (`def n_customers(self) -> int`), a one-liner is enough.

### Exercise 3.1 · Core

The `ConfusionMatrix` below (a compact version of the one from notebooks 01 and 03) works, but has no
hints and no docstrings. Add:

1. a class docstring that says what it represents and lists the four attributes;
2. type hints on `__init__`, `from_predictions` and the three properties (all parameters and return types);
3. a Google-style docstring for `from_predictions` with `Args:`, `Returns:` and `Raises:`
   (it raises `ValueError` when the inputs differ in length);
4. one-line docstrings for `precision` and `recall` that say what they mean *for churn*.

Then run `help(ConfusionMatrix)` to see the result.

In [ ]:
class ConfusionMatrix:
    def __init__(self, tn, fp, fn, tp):
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @classmethod
    def from_predictions(cls, y_true, y_pred):
        t = np.asarray(y_true)
        p = np.asarray(y_pred)
        if t.shape != p.shape:
            raise ValueError(f"y_true and y_pred differ in shape: {t.shape} vs {p.shape}")
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def precision(self):
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self):
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    @property
    def f1(self):
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if p + r else 0.0

In [ ]:
# Check cell for Exercise 3.1
import inspect

assert ConfusionMatrix.__doc__, "add a class docstring"
init_hints = ConfusionMatrix.__init__.__annotations__
assert {"tn", "fp", "fn", "tp", "return"} <= init_hints.keys(), (
    "annotate every __init__ parameter and -> None"
)
fp_hints = ConfusionMatrix.from_predictions.__annotations__
assert {"y_true", "y_pred", "return"} <= fp_hints.keys(), "annotate from_predictions"
doc = inspect.getdoc(ConfusionMatrix.from_predictions) or ""
for section in ("Args:", "Returns:", "Raises:"):
    assert section in doc, f"from_predictions docstring needs a {section} section"
for name in ("precision", "recall", "f1"):
    prop = getattr(ConfusionMatrix, name)
    assert "return" in prop.fget.__annotations__, f"add a return type to {name}"
for name in ("precision", "recall"):
    assert getattr(ConfusionMatrix, name).__doc__, f"add a docstring to {name}"

cm = ConfusionMatrix.from_predictions(y_test, rf.predict(X_test))
print(f"precision {cm.precision:.2f}, recall {cm.recall:.2f}")
print("All checks passed")

<details><summary>Hint</summary>

Arrays and Series can both be passed to `from_predictions`, so a precise but readable hint is
`npt.ArrayLike` from `numpy.typing` (anything NumPy can turn into an array). The return type of a
classmethod that builds an instance is the class itself: `-> "ConfusionMatrix"` (in quotes, because the
class is not finished yet when Python reads the line) or, since Python 3.11, `-> Self` from `typing`.
</details>

<details><summary>Solution</summary>

```python
import numpy.typing as npt


class ConfusionMatrix:
    """Counts of a binary churn classifier's hits and misses (1 = churn).

    Attributes:
        tn: Customers who stayed and were not flagged.
        fp: Customers who stayed but were flagged (wasted offers).
        fn: Churners we did not flag (lost customers).
        tp: Churners we flagged.
    """

    def __init__(self, tn: int, fp: int, fn: int, tp: int) -> None:
        self.tn = tn
        self.fp = fp
        self.fn = fn
        self.tp = tp

    @classmethod
    def from_predictions(cls, y_true: npt.ArrayLike, y_pred: npt.ArrayLike) -> "ConfusionMatrix":
        """Count hits and misses from true labels and predicted labels.

        Args:
            y_true: Observed labels, 1 if the customer churned, else 0.
            y_pred: Predicted labels in the same order as ``y_true``.

        Returns:
            A ConfusionMatrix with the four counts.

        Raises:
            ValueError: If ``y_true`` and ``y_pred`` have different shapes.
        """
        t = np.asarray(y_true)
        p = np.asarray(y_pred)
        if t.shape != p.shape:
            raise ValueError(f"y_true and y_pred differ in shape: {t.shape} vs {p.shape}")
        return cls(
            tn=int(((t == 0) & (p == 0)).sum()),
            fp=int(((t == 0) & (p == 1)).sum()),
            fn=int(((t == 1) & (p == 0)).sum()),
            tp=int(((t == 1) & (p == 1)).sum()),
        )

    @property
    def precision(self) -> float:
        """Share of flagged customers who really churned (how many offers were well spent)."""
        return self.tp / (self.tp + self.fp) if self.tp + self.fp else 0.0

    @property
    def recall(self) -> float:
        """Share of churners we flagged (how many leaving customers we reached)."""
        return self.tp / (self.tp + self.fn) if self.tp + self.fn else 0.0

    @property
    def f1(self) -> float:
        p, r = self.precision, self.recall
        return 2 * p * r / (p + r) if p + r else 0.0
```
</details>

## 4. Dataclasses: structured data without boilerplate

Many classes in data science code mainly *hold data*: a configuration, the result of an experiment, the
schema of a table. Writing `__init__`, `__repr__` and `__eq__` by hand for each of them is repetitive and
easy to get wrong.

The `@dataclass` decorator from the standard library writes those methods for you, based on the
**annotated class attributes**. This is why we needed type hints first: the annotations *are* the field list.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class ExperimentConfig:
    """Settings for one churn training run."""

    model: str = "logistic_regression"
    test_size: float = 0.2
    random_state: int = 42
    threshold: float = 0.5
    features: list[str] = field(
        default_factory=lambda: ["tenure", "MonthlyCharges", "SeniorCitizen"]
    )


config = ExperimentConfig(model="random_forest", threshold=0.3)
print(config)  # generated __repr__
print(config == ExperimentConfig(model="random_forest", threshold=0.3))  # generated __eq__

And the typo from section 1 is now impossible to miss:

In [ ]:
try:
    ExperimentConfig(model="random_forest", n_estimatros=500)
except TypeError as exc:
    print(f"TypeError: {exc}")

### The mutable default trap

Why `field(default_factory=...)` for the feature list instead of simply `features: list[str] = [...]`?
A default value is created **once**, when the class is defined, and shared by every instance. For a list,
that would mean one experiment appending a feature changes the features of every other experiment.
Dataclasses refuse this mistake outright:

In [ ]:
try:

    @dataclass
    class BrokenConfig:
        features: list[str] = ["tenure"]
except ValueError as exc:
    print(f"ValueError: {exc}")

`default_factory` takes a function that is called for every new instance, so each one gets its own list.
(The same trap exists for plain functions: `def f(features=[])` shares one list between all calls. Python
does not warn you there.)

### Validation in `__post_init__`

The generated `__init__` assigns the fields and then calls `__post_init__` if you define one. That is the
place to check values, so an invalid config can never exist.

### Immutability with `frozen=True`

A config describes an experiment that already happened (or is about to). Changing it halfway through a run
makes your logs lie. `frozen=True` makes assignment after creation an error, and as a bonus makes instances
hashable (usable as dict keys).

In [ ]:
from dataclasses import FrozenInstanceError

SUPPORTED_MODELS = ("logistic_regression", "random_forest")


@dataclass(frozen=True)
class ExperimentConfig:
    """Settings for one churn training run.

    Attributes:
        model: Estimator family, one of SUPPORTED_MODELS.
        test_size: Fraction of customers held out for evaluation, in (0, 1).
        random_state: Seed for the split and the estimator.
        threshold: Churn probability from which a customer is flagged, in (0, 1).
        features: Input columns. A tuple, so the frozen config cannot be changed through it.
    """

    model: str = "logistic_regression"
    test_size: float = 0.2
    random_state: int = 42
    threshold: float = 0.5
    features: tuple[str, ...] = ("tenure", "MonthlyCharges", "SeniorCitizen")

    def __post_init__(self) -> None:
        if self.model not in SUPPORTED_MODELS:
            raise ValueError(f"model must be one of {SUPPORTED_MODELS}, got {self.model!r}")
        if not 0 < self.test_size < 1:
            raise ValueError(f"test_size must be between 0 and 1, got {self.test_size}")
        if not 0 < self.threshold < 1:
            raise ValueError(f"threshold must be between 0 and 1, got {self.threshold}")

    @classmethod
    def from_json(cls, path: str | Path) -> "ExperimentConfig":
        """Create a config from a JSON file whose keys are field names."""
        return cls(**json.loads(Path(path).read_text()))


for bad in [{"test_size": 1.5}, {"model": "xgboost"}]:
    try:
        ExperimentConfig(**bad)
    except ValueError as exc:
        print(f"ValueError: {exc}")

config = ExperimentConfig()
try:
    config.threshold = 0.1
except FrozenInstanceError as exc:
    print(f"FrozenInstanceError: {exc}")

### An alternative constructor: loading from a file

The repo contains `configs/baseline.json`. The classmethod `from_json` (remember alternative constructors
from notebook 01) reads it and passes its keys to the normal constructor, so the same validation applies.

In [ ]:
print(CONFIG_PATH.read_text())
config = ExperimentConfig.from_json(CONFIG_PATH)
config

### Dataclasses for results, too

The output of an experiment is also "just data". A frozen dataclass with a computed property gives you a
clean, comparable record. Now a training function needs one argument instead of six.

In [ ]:
@dataclass(frozen=True)
class ExperimentResult:
    """Holdout metrics of one run."""

    config: ExperimentConfig
    precision: float
    recall: float
    n_flagged: int
    n_test: int

    @property
    def flag_rate(self) -> float:
        """Share of test customers the retention team would have to contact."""
        return self.n_flagged / self.n_test


def run_experiment(config: ExperimentConfig, data: pd.DataFrame) -> ExperimentResult:
    """Train on a stratified split and evaluate on the holdout."""
    X = data[list(config.features)]
    y = data["Churn"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=config.test_size, random_state=config.random_state, stratify=y
    )
    if config.model == "logistic_regression":
        model = LogisticRegression(max_iter=1000, random_state=config.random_state)
    else:
        model = RandomForestClassifier(min_samples_leaf=20, random_state=config.random_state)
    model.fit(X_train, y_train)
    flags = (model.predict_proba(X_test)[:, 1] >= config.threshold).astype(int)
    cm = ConfusionMatrix.from_predictions(y_test, flags)
    return ExperimentResult(config, cm.precision, cm.recall, int(flags.sum()), len(y_test))


for cfg in [ExperimentConfig(), ExperimentConfig.from_json(CONFIG_PATH)]:
    result = run_experiment(cfg, customers)
    print(
        f"threshold={cfg.threshold}: precision {result.precision:.2f}, recall {result.recall:.2f}, "
        f"contact {result.flag_rate:.0%} of customers"
    )

Lowering the threshold from 0.5 to 0.3 roughly trades precision for recall: the team reaches more of the
leaving customers but makes more unnecessary offers. With the config as an object, that trade-off is now one
explicit, logged, reproducible setting instead of a number buried in a cell.

The reference package has the full version of this idea in `src/churn/config.py`: more fields, a check that
the target is not used as a feature, and `from_json` that turns a typo in the file into a clear `ConfigError`.

### Check your understanding

```python
@dataclass(frozen=True)
class Config:
    features: list[str] = field(default_factory=list)

c = Config()
c.features.append("tenure")
print(c.features)
```

Does this raise `FrozenInstanceError`?

<details><summary>Answer</summary>

No, it prints `['tenure']`. `frozen=True` only blocks *assigning* a new value to a field
(`c.features = [...]`). It does not make the list inside immutable. That is why the configs above and in
`src/churn` use tuples for feature lists: a frozen dataclass is only really frozen if its fields are
immutable too.
</details>

### Exercise 4.1 · Core

A colleague keeps their random forest settings in a dict:

```python
raw_config = {"model": "random_forest", "n_estimators": 300, "test_size": 0.2,
              "threshold": 0.3, "random_state": 42}
```

Write a frozen dataclass `TrainingConfig` with these five fields and these defaults:
`model="random_forest"`, `n_estimators=100`, `test_size=0.2`, `threshold=0.5`, `random_state=42`.
In `__post_init__`, raise a `ValueError` with a helpful message if

- `model` is not `"logistic_regression"` or `"random_forest"`,
- `n_estimators` is not positive,
- `test_size` or `threshold` is not strictly between 0 and 1.

Add a short class docstring that explains the fields.

In [ ]:
@dataclass(frozen=True)
class TrainingConfig: ...  # Your code here

In [ ]:
# Check cell for Exercise 4.1
raw_config = {
    "model": "random_forest",
    "n_estimators": 300,
    "test_size": 0.2,
    "threshold": 0.3,
    "random_state": 42,
}
cfg = TrainingConfig(**raw_config)
assert cfg.n_estimators == 300 and cfg.threshold == 0.3
assert TrainingConfig() == TrainingConfig(
    model="random_forest", n_estimators=100, test_size=0.2, threshold=0.5, random_state=42
)
assert not TrainingConfig.__doc__.startswith("TrainingConfig("), "add a class docstring"

for bad in [{"model": "xgboost"}, {"n_estimators": 0}, {"test_size": 1.5}, {"threshold": 1.0}]:
    try:
        TrainingConfig(**bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"{bad} should be rejected")

try:
    TrainingConfig(n_estimatros=500)
except TypeError:
    pass
else:
    raise AssertionError("a typo in a field name should raise TypeError")

try:
    cfg.threshold = 0.1
except FrozenInstanceError:
    pass
else:
    raise AssertionError("TrainingConfig should be frozen")
print("All checks passed")

<details><summary>Hint</summary>

Copy the structure of `ExperimentConfig` above. Note that `@dataclass` writes its own docstring if you do
not give one (something like `TrainingConfig(model: str = ...)`), which is why the check looks at the text.
</details>

<details><summary>Solution</summary>

```python
@dataclass(frozen=True)
class TrainingConfig:
    """Settings for training a churn classifier.

    Attributes:
        model: "logistic_regression" or "random_forest".
        n_estimators: Number of trees (used by the random forest only), > 0.
        test_size: Fraction of customers held out for evaluation, in (0, 1).
        threshold: Churn probability from which a customer is flagged, in (0, 1).
        random_state: Seed for the split and the estimator.
    """

    model: str = "random_forest"
    n_estimators: int = 100
    test_size: float = 0.2
    threshold: float = 0.5
    random_state: int = 42

    def __post_init__(self) -> None:
        if self.model not in ("logistic_regression", "random_forest"):
            raise ValueError(f"unknown model {self.model!r}")
        if self.n_estimators <= 0:
            raise ValueError(f"n_estimators must be positive, got {self.n_estimators}")
        if not 0 < self.test_size < 1:
            raise ValueError(f"test_size must be between 0 and 1, got {self.test_size}")
        if not 0 < self.threshold < 1:
            raise ValueError(f"threshold must be between 0 and 1, got {self.threshold}")
```
</details>

### Exercise 4.2 · Stretch

Add a classmethod `from_json(path)` to your `TrainingConfig`. When the file contains a key that is not a
field (a typo such as `"tresh0ld"`), raise a `ValueError` whose message **names the unknown key(s)**,
instead of Python's generic `TypeError`.

Tip: `dataclasses.fields(cls)` lists the fields of a dataclass.

Write the complete class again in the cell below (you can copy your solution of 4.1).

In [ ]:
@dataclass(frozen=True)
class TrainingConfig: ...  # Your code here (copy 4.1 and add from_json)

In [ ]:
# Check cell for Exercise 4.2
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    good = Path(tmp) / "good.json"
    good.write_text(json.dumps({"model": "logistic_regression", "threshold": 0.3}))
    cfg = TrainingConfig.from_json(good)
    assert cfg.model == "logistic_regression" and cfg.threshold == 0.3

    typo = Path(tmp) / "typo.json"
    typo.write_text(json.dumps({"tresh0ld": 0.3, "n_estimators": 50}))
    try:
        TrainingConfig.from_json(typo)
    except ValueError as exc:
        assert "tresh0ld" in str(exc), "name the unknown key in the message"
        print(f"Good error message: {exc}")
    else:
        raise AssertionError("an unknown key should raise ValueError")
print("All checks passed")

<details><summary>Solution</summary>

```python
from dataclasses import fields


@dataclass(frozen=True)
class TrainingConfig:
    """Settings for training a churn classifier (see Exercise 4.1)."""

    model: str = "random_forest"
    n_estimators: int = 100
    test_size: float = 0.2
    threshold: float = 0.5
    random_state: int = 42

    def __post_init__(self) -> None:
        if self.model not in ("logistic_regression", "random_forest"):
            raise ValueError(f"unknown model {self.model!r}")
        if self.n_estimators <= 0:
            raise ValueError(f"n_estimators must be positive, got {self.n_estimators}")
        if not 0 < self.test_size < 1:
            raise ValueError(f"test_size must be between 0 and 1, got {self.test_size}")
        if not 0 < self.threshold < 1:
            raise ValueError(f"threshold must be between 0 and 1, got {self.threshold}")

    @classmethod
    def from_json(cls, path: str | Path) -> "TrainingConfig":
        """Create a config from a JSON file.

        Raises:
            ValueError: If the file contains keys that are not config fields,
                or if a value is invalid.
        """
        raw = json.loads(Path(path).read_text())
        allowed = {f.name for f in fields(cls)}
        unknown = sorted(set(raw) - allowed)
        if unknown:
            raise ValueError(f"Unknown keys in {path}: {unknown}. Allowed: {sorted(allowed)}")
        return cls(**raw)
```

`src/churn/config.py` solves the same problem differently: it lets the `TypeError` happen and re-raises it
as a `ConfigError` with `raise ... from exc`. Both are fine; the explicit check gives a nicer message.
</details>

## 5. Pydantic: validation at the boundary

Dataclasses check nothing about types: `TrainingConfig(n_estimators="300")` is accepted and only fails later,
inside scikit-learn. For data that comes **from outside your program** (a YAML or JSON file written by a
person, an API request, a message from another service), you usually want stronger guarantees. That is what
**Pydantic** is for: you declare fields with type hints and constraints, and Pydantic validates (and, where it
is safe, converts) every value when the object is created.

In [ ]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator


class PydanticTrainingConfig(BaseModel):
    """Training settings, validated when loaded from a config file."""

    model_config = ConfigDict(
        frozen=True, extra="forbid"
    )  # Pydantic's own settings, not our "model"

    model: Literal["logistic_regression", "random_forest"] = "random_forest"
    n_estimators: int = Field(100, gt=0, le=5000)
    test_size: float = Field(0.2, gt=0, lt=1)
    threshold: float = Field(0.5, gt=0, lt=1)
    random_state: int = 42

    @field_validator("threshold")
    @classmethod
    def warn_about_extreme_thresholds(cls, value: float) -> float:
        if value < 0.05:
            raise ValueError("a threshold below 0.05 flags almost every customer")
        return value


cfg = PydanticTrainingConfig(n_estimators="300", threshold=0.3)  # "300" is converted to 300
print(cfg)
print(cfg.model_dump())

In [ ]:
try:
    PydanticTrainingConfig(model="xgboost", test_size=1.5, tresh0ld=0.3)
except ValidationError as exc:
    print(exc)

Notice that Pydantic reports **all** problems at once, each with the field name and the reason. That is
exactly what you want to show a person who just edited a config file.

### Dataclass or Pydantic?

| Situation | Choose |
|---|---|
| Data created by your own code (results, internal records) | dataclass |
| Config with a few range checks, no extra dependency wanted | dataclass + `__post_init__` |
| Untrusted input: config files edited by hand, API requests, JSON from other systems | Pydantic |
| You need type conversion ("300" to 300, ISO strings to dates) or JSON schema | Pydantic |

A common architecture: validate with Pydantic **at the boundary** where data enters (loading the file, the
API endpoint), then pass plain, trusted objects to the rest of the code.

### Exercise 5.1 · Challenge

1. Write `PydanticExperimentConfig` with the fields of the frozen `ExperimentConfig` from section 4
   (`model`, `test_size`, `random_state`, `threshold`, `features` as a tuple of strings), using `Literal`
   and `Field` constraints instead of `__post_init__`. Forbid extra keys and make it frozen.
2. Add a validator that rejects an empty `features` tuple and one that rejects `"Churn"` as a feature
   (it is the target: using it would be perfect, useless "leakage").
3. Load `../configs/baseline.json` with `PydanticExperimentConfig.model_validate_json(...)`.
4. Discussion (write 3-4 sentences in a markdown cell): the reference package validates in
   `ExperimentConfig.__post_init__`. Where else could validation live (the CLI, the loader, the model), and
   what goes wrong if every layer re-validates everything?

In [ ]:
class PydanticExperimentConfig(BaseModel): ...  # Your code here

In [ ]:
# Check cell for Exercise 5.1
cfg = PydanticExperimentConfig.model_validate_json(CONFIG_PATH.read_text())
assert cfg.threshold == 0.3 and cfg.model == "logistic_regression"
assert isinstance(cfg.features, tuple) and len(cfg.features) > 0

for bad in [
    {"test_size": 0},
    {"model": "xgboost"},
    {"features": ()},
    {"features": ("tenure", "Churn")},
    {"tresh0ld": 0.3},
]:
    try:
        PydanticExperimentConfig(**bad)
    except ValidationError:
        pass
    else:
        raise AssertionError(f"{bad} should be rejected")

try:
    cfg.threshold = 0.1
except ValidationError:
    pass
else:
    raise AssertionError("the config should be frozen")
print("All checks passed")

<details><summary>Solution</summary>

```python
class PydanticExperimentConfig(BaseModel):
    """Experiment settings loaded from a JSON file."""

    model_config = ConfigDict(frozen=True, extra="forbid")

    model: Literal["logistic_regression", "random_forest"] = "logistic_regression"
    test_size: float = Field(0.2, gt=0, lt=1)
    random_state: int = 42
    threshold: float = Field(0.5, gt=0, lt=1)
    features: tuple[str, ...] = ("tenure", "MonthlyCharges", "SeniorCitizen")

    @field_validator("features")
    @classmethod
    def check_features(cls, value: tuple[str, ...]) -> tuple[str, ...]:
        if not value:
            raise ValueError("at least one feature is required")
        if "Churn" in value:
            raise ValueError("'Churn' is the target and cannot be a feature")
        return value
```

Discussion points: validate once, where data enters the system (loading the config, reading a batch). Inner
code can then trust its inputs and stay simple. If every layer re-checks everything, the rules get duplicated,
drift apart over time ("the CLI allows 0.9, the model allows 0.95"), and nobody knows which check is the real
one. Some checks do belong deeper, though: `ChurnModel` checks that the columns it needs are present, because
only the model knows which columns those are.
</details>

## Summary

- Type hints document what a function expects and returns. Python ignores them at runtime; your editor and
  `mypy` use them to find bugs before the code runs.
- Write `X | None = None` whenever `None` is allowed. `list[str] = None` is a lie.
- Docstrings (Google style: summary line, `Args:`, `Returns:`, `Raises:`) explain meaning, units and ranges.
  `help()` and your editor show them.
- `@dataclass` generates `__init__`, `__repr__` and `__eq__` from annotated fields. Use
  `field(default_factory=...)` for lists and dicts, `__post_init__` for validation, `frozen=True` for
  configs and results.
- A config object turns a silent typo into an immediate `TypeError`, and invalid values into an error at
  creation time.
- Pydantic validates and converts untrusted input at the boundary of your system.

**Common pitfalls:** a mutable default (`def f(x=[])`); thinking `frozen=True` freezes the list inside;
validating the same thing in five places; docstrings that repeat the function name ("Computes recall.
Returns the recall.") instead of explaining meaning.

## Going further

- `typing.TypeAlias` (or the `type` statement in Python 3.12) gives a complex type a name:
  `type Metrics = dict[str, list[float]]`.
- `typing.TYPE_CHECKING` lets you import a module only for type hints, avoiding import cycles.
- `typing.Protocol` describes "anything with these methods" (notebook 05); used throughout `src/churn`.
- `mypy --strict` turns on every check. Try it on `src/churn` and see what it asks for.
- `@dataclass(slots=True)` makes instances smaller and faster; `@dataclass(order=True)` generates `<`, but
  think twice before making results "orderable" (notebook 03).

Resources:

- Python docs, dataclasses: https://docs.python.org/3/library/dataclasses.html
- mypy cheat sheet: https://mypy.readthedocs.io/en/stable/cheat_sheet_py3.html
- Google Python style guide, docstrings: https://google.github.io/styleguide/pyguide.html#38-comments-and-docstrings
- Pydantic documentation: https://docs.pydantic.dev/latest/